# 01 - Data overview

**Purpose:** Summarise the real datasets read-only and chart them (Track C is descriptive only - no model).

**Inputs:** CESNET-TLS22 S (HDF5) and banking_pqc_observations.csv (dataset env vars / default pack path)

**Outputs:** status-share chart by scan date, counts chart, CESNET rows-per-day chart

**Status labels:** VERIFIED (real public data, read-only). Not bank data; ERROR stays unclassified; repeated hosts across dates are kept.

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
from cryptoshift.data_summary import build_data_summary
s = build_data_summary(chunk_size=20_000, max_stream_rows=40_000)
pqc, ces = s['banking_pqc_observations'], s['cesnet_tls22_s']
print('Banking observations:', pqc['row_count'], 'rows |', len(pqc['scan_dates']), 'scan dates | duplicate keys', pqc['duplicate_observation_keys'], '|', pqc['unique_hosts'], 'hosts')
print('CESNET-TLS22 S:', f"{ces['total_rows']:,}", 'rows in', ces['table_count'], 'daily tables |', ' to '.join(ces['date_range']))

In [ ]:
counts = pd.DataFrame(pqc['status_counts_by_scan_date']).T.fillna(0)[['PQ-SAFE','CLASSICAL','ERROR']].astype(int)
share = counts.div(counts.sum(axis=1), axis=0)
colors = {'PQ-SAFE':'#2a9d8f','CLASSICAL':'#e76f51','ERROR':'#9e9e9e'}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts.plot(kind='bar', stacked=True, color=[colors[c] for c in counts.columns], ax=ax[0], title='Observations per scan date (count)')
share.plot(kind='line', marker='o', color=[colors[c] for c in share.columns], ax=ax[1], title='Share of the 64 banking hosts')
ax[1].set_ylim(0, 1); ax[0].set_xlabel('scan date'); ax[1].set_xlabel('scan date')
savefig('01_pqc_status_by_date')
counts

In [ ]:
rows = pd.Series({t.split('/')[-1][1:]: n for t, n in ces['table_rows'].items()})
ax = rows.plot(kind='bar', figsize=(11, 3.5), color='#457b9d', title='CESNET-TLS22 S: flows per daily table (Oct 2021)')
ax.set_ylabel('flows'); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M'))
savefig('01_cesnet_rows_per_day')

Public-endpoint observations on the given dates only. They say nothing about a bank's internal systems, signatures or migration status. CESNET is real 2021 traffic metadata used only for scale/streaming demos.